# Blog Post - AI Fairness 360: Mitigating Bias in Machine Learning Models

# BLOG V2

### Title
**Creating Dataset Objects in AI Fairness 360 and Exploring Fairness Metrics**

### Introduction

Ensuring fairness in machine learning models is a crucial step towards building ethical AI systems. AI Fairness 360 (AIF360) is an open-source toolkit developed by IBM to help data scientists detect and mitigate bias in their models. In this blog post, we will focus on creating dataset objects using both the `BinaryLabelDataset` and `StandardDataset` classes provided by AIF360. Additionally, we will explore some key fairness metrics to understand how bias can be measured and addressed.

### Lesson Objectives

By the end of this lesson, you will be able to:
- Create dataset objects using `BinaryLabelDataset` and `StandardDataset` classes.
- Understand the differences and use cases for each dataset class.
- Utilize fairness metrics to evaluate bias in datasets and models.

### Prerequisites

Before we begin, ensure you have the following installed:
- Python 3.x
- Pandas
- AI Fairness 360

You can install AI Fairness 360 using pip:
```bash
pip install aif360
```

### Creating Dataset Objects

#### Loading the Dataset

We will use the Recidivism for Offenders Released from Prison dataset, which provides information on individuals released from prison and their likelihood of reoffending. Download the dataset from Kaggle and load it into a pandas DataFrame.


In [8]:

import pandas as pd
pd.set_option("display.max_columns", 100)
df = pd.read_csv("blog_post/data/Iowa_Prison_Recidivism_Status_20240724.csv", 
                 index_col=0, usecols=range(0, 23-7))   

## Quick Conversion of Dtypes for Clean Data
df = df.convert_dtypes(convert_string=False)

# Drop unnecessary columns
drop_cols = ['Supervising Unit','Supervision Start Date','Supervision End Date']
df = df.drop(columns=drop_cols)
df.info()
df.head()



<class 'pandas.core.frame.DataFrame'>
Index: 25244 entries, 20655350 to 20999813
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Race                         25244 non-null  object 
 1   Sex                          25241 non-null  object 
 2   Age                          25244 non-null  Int64  
 3   Supervision Type             25244 non-null  object 
 4   Cohort Fiscal Year           25244 non-null  Int64  
 5   Report Fiscal Year           25244 non-null  Int64  
 6   Months Supervised            25244 non-null  Int64  
 7   Supervision End Reason       25244 non-null  object 
 8   Supervision Offense Class    25244 non-null  object 
 9   Supervision Offense Type     25244 non-null  object 
 10  Supervision Offense Subtype  25244 non-null  object 
 11  Reincarcerated               25244 non-null  boolean
dtypes: Int64(4), boolean(1), object(7)
memory usage: 2.5+ MB


,Race,Sex,Age,Supervision Type,Cohort Fiscal Year,Report Fiscal Year,Months Supervised,Supervision End Reason,Supervision Offense Class,Supervision Offense Type,Supervision Offense Subtype,Reincarcerated
Offender Number,,,,,,,,,,,,
20655350,White,Male,35,Prison,2018,2021,125,Discharged - Expiration of Sentence,B Felony,Violent,Sex,False
18876932,Black,Male,30,Prison,2017,2020,49,Released to Special Sentence,D Felony,Other,Other Criminal,False
2424146,White,Male,40,Prison,2016,2019,8,Parole Granted,Aggravated Misdemeanor,Public Order,Other Public Order,False
19088303,Black,Male,29,Work Release,2016,2019,2,Parole Granted,C Felony,Drug,Trafficking,False
20280797,White,Male,38,Prison,2016,2019,11,Paroled w/Immediate Discharge,Aggravated Misdemeanor,Property,Burglary,False



#### Encoding Categorical Features

To work with AIF360, we need to encode the categorical features numerically. In this example, we'll focus on the 'race' attribute as a protected attribute.


In [9]:
df['Race'].unique()

array(['White', 'Black', 'Hispanic', 'Asian or Pacific Islander',
       'American Indian or Alaska Native', 'Unknown'], dtype=object)

In [10]:

# Encode the 'race' column
race_map = {'White': 0, 'Black': 1, 'Hispanic': 2, 'Asian or Pacific Islander': 3,
            'American Indian or Alaska Native': 4, 'Unknown': 5, 'Other':6}
df['Race'] = df['Race'].map(race_map)

# Encode the "Sex" column
sex_map = {"Male": 0, "Female":1}
df['Sex'] = df['Sex'].map(sex_map)
df

,Race,Sex,Age,Supervision Type,Cohort Fiscal Year,Report Fiscal Year,Months Supervised,Supervision End Reason,Supervision Offense Class,Supervision Offense Type,Supervision Offense Subtype,Reincarcerated
Offender Number,,,,,,,,,,,,
20655350,0,0.0,35,Prison,2018,2021,125,Discharged - Expiration of Sentence,B Felony,Violent,Sex,False
18876932,1,0.0,30,Prison,2017,2020,49,Released to Special Sentence,D Felony,Other,Other Criminal,False
2424146,0,0.0,40,Prison,2016,2019,8,Parole Granted,Aggravated Misdemeanor,Public Order,Other Public Order,False
19088303,1,0.0,29,Work Release,2016,2019,2,Parole Granted,C Felony,Drug,Trafficking,False
20280797,0,0.0,38,Prison,2016,2019,11,Paroled w/Immediate Discharge,Aggravated Misdemeanor,Property,Burglary,False
...,...,...,...,...,...,...,...,...,...,...,...,...
20997251,0,0.0,30,Prison,2020,2023,2,Paroled w/Immediate Discharge,D Felony,Public Order,Weapons,False
20997488,0,0.0,36,Work Release,2020,2023,5,Parole Granted,D Felony,Drug,Drug Possession,False
20999039,0,0.0,27,Prison,2020,2023,31,Parole Granted,C Felony,Property,Burglary,True


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25244 entries, 20655350 to 20999813
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Race                         25244 non-null  int64  
 1   Sex                          25241 non-null  float64
 2   Age                          25244 non-null  Int64  
 3   Supervision Type             25244 non-null  object 
 4   Cohort Fiscal Year           25244 non-null  Int64  
 5   Report Fiscal Year           25244 non-null  Int64  
 6   Months Supervised            25244 non-null  Int64  
 7   Supervision End Reason       25244 non-null  object 
 8   Supervision Offense Class    25244 non-null  object 
 9   Supervision Offense Type     25244 non-null  object 
 10  Supervision Offense Subtype  25244 non-null  object 
 11  Reincarcerated               25244 non-null  boolean
dtypes: Int64(4), boolean(1), float64(1), int64(1), object(5)
memory usage

In [12]:
cat_cols = df.select_dtypes(include='object').columns
cat_cols

Index(['Supervision Type', 'Supervision End Reason',
       'Supervision Offense Class', 'Supervision Offense Type',
       'Supervision Offense Subtype'],
      dtype='object')

In [14]:
# Impute NaNs with most frequent value
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn import set_config
set_config(transform_output='pandas')

# Categorical Pipeline
cat_cols = df.select_dtypes(include='object').columns
cat_imputer = SimpleImputer(strategy='most_frequent')
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_pipe = make_pipeline(cat_imputer, ohe)


# Convert Boolean Columns to Integers
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# Numeric Pipeline
num_cols = df.select_dtypes(include='number').columns
num_imputer = SimpleImputer(strategy='mean')
num_pipe = make_pipeline(num_imputer)

preprocessor = ColumnTransformer(transformers=[('cat', cat_pipe, cat_cols),
                                               ('num', num_pipe, num_cols)],
                                 remainder='passthrough',
                                 verbose_feature_names_out=False)
# imputer.fit_transform(df[cat_cols])
final_df = preprocessor.fit_transform(df).reset_index()
final_df

,Offender Number,Supervision Type_Prison,Supervision Type_Work Release,Supervision End Reason_Discharged - Expiration of Sentence,Supervision End Reason_Parole Granted,Supervision End Reason_Paroled to Detainer - INS,Supervision End Reason_Paroled to Detainer - Iowa,Supervision End Reason_Paroled to Detainer - Out of State,Supervision End Reason_Paroled to Detainer - U.S. Marshall,Supervision End Reason_Paroled w/Immediate Discharge,Supervision End Reason_Released to Special Sentence,Supervision Offense Class_A Felony,Supervision Offense Class_Aggravated Misdemeanor,Supervision Offense Class_B Felony,Supervision Offense Class_C Felony,Supervision Offense Class_D Felony,Supervision Offense Class_Felony - Enhancement to Original Penalty,Supervision Offense Class_Felony - Mandatory Minimum,Supervision Offense Class_Other Felony,Supervision Offense Class_Serious Misdemeanor,Supervision Offense Class_Simple Misdemeanor,Supervision Offense Class_Special Sentence 2005,Supervision Offense Type_Drug,Supervision Offense Type_Other,Supervision Offense Type_Property,Supervision Offense Type_Public Order,Supervision Offense Type_Violent,Supervision Offense Subtype_Alcohol,Supervision Offense Subtype_Animals,Supervision Offense Subtype_Arson,Supervision Offense Subtype_Assault,Supervision Offense Subtype_Burglary,Supervision Offense Subtype_Drug Possession,Supervision Offense Subtype_Flight/Escape,Supervision Offense Subtype_Forgery/Fraud,Supervision Offense Subtype_Kidnap,Supervision Offense Subtype_Murder/Manslaughter,Supervision Offense Subtype_OWI,Supervision Offense Subtype_Other Criminal,Supervision Offense Subtype_Other Drug,Supervision Offense Subtype_Other Government,Supervision Offense Subtype_Other Public Order,Supervision Offense Subtype_Other Violent,Supervision Offense Subtype_Prostitution/Pimping,Supervision Offense Subtype_Robbery,Supervision Offense Subtype_Sex,Supervision Offense Subtype_Stolen Property,Supervision Offense Subtype_Theft,Supervision Offense Subtype_Traffic,Supervision Offense Subtype_Trafficking,Supervision Offense Subtype_Vandalism,Supervision Offense Subtype_Weapons,Race,Sex,Age,Cohort Fiscal Year,Report Fiscal Year,Months Supervised,Reincarcerated
0,20655350,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0,2018.0,2021.0,125.0,0.0
1,18876932,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,30.0,2017.0,2020.0,49.0,0.0
2,2424146,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,2016.0,2019.0,8.0,0.0
3,19088303,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,29.0,2016.0,2019.0,2.0,0.0
4,20280797,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,2016.0,2019.0,11.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25239,20997251,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,30.0,2020.0,2023.0,2.0,0.0
25240,20997488,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0


### Creating a `BinaryLabelDataset`

The `BinaryLabelDataset` is tailored for binary classification tasks where the target variable has two possible outcomes.


In [19]:
# from aif360.datasets import BinaryLabelDataset

# # Create a BinaryLabelDataset
# binary_dataset = BinaryLabelDataset(
#     df = final_df,
#     label_names=['Reincarcerated'],
#     # favorable_classes=[0],  # Non-recidivism is favorable
#     favorable_label=0,  # Non-recidivism is favorable
#     protected_attribute_names=['Race','Sex'],
#     # privileged_classes=[[0],[0]]  # Privileged group: White
# )

# # Print dataset details
# print(binary_dataset.feature_names)
# print(binary_dataset.label_names)
# print(binary_dataset.protected_attribute_names)




### Creating a `StandardDataset`

The `StandardDataset` is more flexible and can handle binary, multi-class, and continuous labels.


In [21]:

from aif360.datasets import StandardDataset

# Create a StandardDataset
standard_dataset = StandardDataset(
    final_df,
    label_name='Reincarcerated', # Recidivism is the label
    favorable_classes=[0], # Non-recidivism is favorable
    protected_attribute_names=['Race','Sex'],  # List of the protected attributes
    privileged_classes=[[race_map['White']],[sex_map['Male']]]  # Privileged group: White
)

# Print dataset details
print(standard_dataset.feature_names)
print(standard_dataset.label_names)
print(standard_dataset.protected_attribute_names)


['Offender Number', 'Supervision Type_Prison', 'Supervision Type_Work Release', 'Supervision End Reason_Discharged - Expiration of Sentence', 'Supervision End Reason_Parole Granted', 'Supervision End Reason_Paroled to Detainer - INS', 'Supervision End Reason_Paroled to Detainer - Iowa', 'Supervision End Reason_Paroled to Detainer - Out of State', 'Supervision End Reason_Paroled to Detainer - U.S. Marshall', 'Supervision End Reason_Paroled w/Immediate Discharge', 'Supervision End Reason_Released to Special Sentence', 'Supervision Offense Class_A Felony', 'Supervision Offense Class_Aggravated Misdemeanor', 'Supervision Offense Class_B Felony', 'Supervision Offense Class_C Felony', 'Supervision Offense Class_D Felony', 'Supervision Offense Class_Felony - Enhancement to Original Penalty', 'Supervision Offense Class_Felony - Mandatory Minimum', 'Supervision Offense Class_Other Felony', 'Supervision Offense Class_Serious Misdemeanor', 'Supervision Offense Class_Simple Misdemeanor', 'Supervis



### Exploring Fairness Metrics

AI Fairness 360 provides several fairness metrics to evaluate bias in datasets and models. We'll explore some key metrics using both `BinaryLabelDataset` and `StandardDataset`.

#### Disparate Impact

Disparate impact is a ratio that measures the relative likelihood of a favorable outcome between unprivileged and privileged groups.


In [23]:

from aif360.metrics import BinaryLabelDatasetMetric

# # Disparate Impact for BinaryLabelDataset
# binary_metric = BinaryLabelDatasetMetric(
#     binary_dataset,
#     privileged_groups=[{'Race': 0}],
#     unprivileged_groups=[{"Race":i} for i in range(1,len(race_map))]
# )
# print(f"Disparate Impact (BinaryLabelDataset): {binary_metric.disparate_impact()}")


In [24]:

# Disparate Impact for StandardDataset
standard_metric = BinaryLabelDatasetMetric(
    standard_dataset,
    privileged_groups=[{'Race': 0}],
    unprivileged_groups=[{"Race":i} for i in range(1,len(race_map))]
)
print(f"Disparate Impact (StandardDataset): {standard_metric.disparate_impact()}")


Disparate Impact (StandardDataset): 1.0561530353156547



#### Statistical Parity Difference

Statistical parity difference measures the difference in the probability of favorable outcomes between unprivileged and privileged groups.


In [26]:

# Statistical Parity Difference for BinaryLabelDataset
# print(f"Statistical Parity Difference (BinaryLabelDataset): {binary_metric.statistical_parity_difference()}")

# Statistical Parity Difference for StandardDataset
print(f"Statistical Parity Difference (StandardDataset): {standard_metric.statistical_parity_difference()}")

Statistical Parity Difference (StandardDataset): 0.034476031755028



#### Equal Opportunity Difference

Equal opportunity difference measures the difference in true positive rates between unprivileged and privileged groups.


In [27]:

from aif360.metrics import ClassificationMetric

# Assuming you have a model's predictions for demonstration purposes
# Example: true_labels and predicted_labels are arrays of true and predicted labels

true_labels = binary_dataset.labels
predicted_labels = binary_dataset.labels  # Placeholder, replace with actual model predictions

# Create a ClassificationMetric object
classification_metric = ClassificationMetric(
    binary_dataset,
    binary_dataset,  # Use the same dataset for demonstration
    unprivileged_groups=[{'race': 0}],
    privileged_groups=[{'race': 1}]
)

# Calculate Equal Opportunity Difference
print(f"Equal Opportunity Difference: {classification_metric.equal_opportunity_difference()}")


NameError: name 'binary_dataset' is not defined


### Conclusion

In this lesson, we explored how to create dataset objects using both the `BinaryLabelDataset` and `StandardDataset` classes in AI Fairness 360. We also discussed some key fairness metrics to evaluate bias in datasets and models. By leveraging these tools, you can take a significant step towards building fairer and more ethical AI systems.

### Further Reading
- IBM AI Fairness 360: [AIF360 Documentation](https://aif360.mybluemix.net/)
- Fairness and Machine Learning by Solon Barocas, Moritz Hardt, and Arvind Narayanan
- Algorithmic Bias Detection and Mitigation: Best Practices and Policies to Reduce Consumer Harms by the Federal Trade Commission

### Call to Action
I encourage you to experiment with AI Fairness 360 in your projects and contribute to creating fair AI systems. Together, we can make AI work for everyone.

### References
- [Disparate Impact](https://en.wikipedia.org/wiki/Disparate_impact)
- [AI Fairness 360 GitHub Repository](https://github.com/IBM/AIF360)
- [Bias in AI: A Review of Literature](https://arxiv.org/abs/1908.09635)

---

Feel free to customize or expand any sections to better fit your style and the level of detail you want to provide. If you have any specific requests or need further elaboration on any part, let me know!

No, each dataset in AI Fairness 360 can have multiple protected attributes. This allows you to analyze and mitigate bias across various dimensions simultaneously. Both `BinaryLabelDataset` and `StandardDataset` support multiple protected attributes.



### Adding Multiple Protected Attributes



Here’s how you can specify multiple protected attributes for both `BinaryLabelDataset` and `StandardDataset`.



#### Example with `BinaryLabelDataset`



Let's say you want to include both `race` and `gender` as protected attributes:



```python
from aif360.datasets import BinaryLabelDataset

# Encode 'race' and 'gender'
df['race'] = df['race'].map({'Black': 0, 'White': 1, 'Other': 2})
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# Create a BinaryLabelDataset with multiple protected attributes
binary_dataset = BinaryLabelDataset(
    df,
    label_name='recidivism',
    favorable_classes=[0],  # Non-recidivism is favorable
    protected_attribute_names=['race', 'gender'],
    privileged_classes=[[1], [1]]  # Privileged groups: White and Male
)

# Print dataset details
print(binary_dataset.feature_names)
print(binary_dataset.label_names)
print(binary_dataset.protected_attribute_names)
```



#### Example with `StandardDataset`

Similarly, for `StandardDataset`, you can specify multiple protected attributes:

```python
from aif360.datasets import StandardDataset

# Create a StandardDataset with multiple protected attributes
standard_dataset = StandardDataset(
    df,
    label_name='recidivism',
    favorable_classes=[0],
    protected_attribute_names=['race', 'gender'],
    privileged_classes=[[1], [1]]  # Privileged groups: White and Male
)

# Print dataset details
print(standard_dataset.feature_names)
print(standard_dataset.label_names)
print(standard_dataset.protected_attribute_names)
```

### Exploring Fairness Metrics with Multiple Protected Attributes

When using multiple protected attributes, you can still calculate fairness metrics. Here’s how you can do it:

#### Disparate Impact

Calculate disparate impact considering multiple protected attributes:

```python
from aif360.metrics import BinaryLabelDatasetMetric

# Disparate Impact for BinaryLabelDataset
binary_metric = BinaryLabelDatasetMetric(
    binary_dataset,
    privileged_groups=[{'race': 1, 'gender': 1}],
    unprivileged_groups=[{'race': 0, 'gender': 0}]
)
print(f"Disparate Impact (BinaryLabelDataset): {binary_metric.disparate_impact()}")

# Disparate Impact for StandardDataset
standard_metric = BinaryLabelDatasetMetric(
    standard_dataset,
    privileged_groups=[{'race': 1, 'gender': 1}],
    unprivileged_groups=[{'race': 0, 'gender': 0}]
)
print(f"Disparate Impact (StandardDataset): {standard_metric.disparate_impact()}")
```

#### Statistical Parity Difference

Calculate statistical parity difference considering multiple protected attributes:

```python
# Statistical Parity Difference for BinaryLabelDataset
print(f"Statistical Parity Difference (BinaryLabelDataset): {binary_metric.statistical_parity_difference()}")

# Statistical Parity Difference for StandardDataset
print(f"Statistical Parity Difference (StandardDataset): {standard_metric.statistical_parity_difference()}")
```

#### Equal Opportunity Difference

Calculate equal opportunity difference considering multiple protected attributes:

```python
from aif360.metrics import ClassificationMetric

# Assuming you have a model's predictions for demonstration purposes
true_labels = binary_dataset.labels
predicted_labels = binary_dataset.labels  # Placeholder, replace with actual model predictions

# Create a ClassificationMetric object
classification_metric = ClassificationMetric(
    binary_dataset,
    binary_dataset,  # Use the same dataset for demonstration
    unprivileged_groups=[{'race': 0, 'gender': 0}],
    privileged_groups=[{'race': 1, 'gender': 1}]
)

# Calculate Equal Opportunity Difference
print(f"Equal Opportunity Difference: {classification_metric.equal_opportunity_difference()}")
```



### Conclusion

In summary, AI Fairness 360 allows you to handle multiple protected attributes in both `BinaryLabelDataset` and `StandardDataset`. This capability is crucial for analyzing and mitigating bias across various dimensions, ensuring a more comprehensive approach to fairness in AI systems.

Feel free to expand on any sections or let me know if you need further details or specific explanations!

# BLOG V1

## Title: **AI Fairness 360: Ensuring Equity in Machine Learning**



### Introduction

Artificial Intelligence (AI) has revolutionized various sectors, from healthcare to finance, by providing powerful tools to analyze data and make informed decisions. However, as these AI systems become more integrated into society, ensuring that they operate fairly and without bias becomes increasingly important. Bias in AI can lead to unfair outcomes, which can have significant social and ethical implications.

To address this issue, IBM developed AI Fairness 360 (AIF360), an open-source toolkit designed to help developers detect and mitigate bias in machine learning models. In this lesson, we will explore what AI Fairness 360 is, why fairness in AI is crucial, and how to use the toolkit to create more equitable AI systems.

### Blog Objectives

By the end of this post, you will be able to:

- Understand the concept of bias in AI and its implications.
- Learn how to use AI Fairness 360 to detect and mitigate bias.
- Gain hands-on experience with a practical example using a real-world dataset.




### Understanding Bias in AI

As we delve into the topic of fairness in AI, it's essential to first understand what bias is and how it can manifest in machine learning systems.

**Types of Bias**
Bias in AI can stem from various sources and can be broadly categorized into three types:

1. **Data Bias**: This occurs when the training data used to build the model is not representative of the real-world population. For example, if a facial recognition system is trained predominantly on images of lighter-skinned individuals, it may perform poorly on individuals with darker skin tones.

2. **Algorithmic Bias**: This arises when the algorithms used in the AI system inadvertently amplify existing biases in the data. Even if the training data is balanced, the way the algorithm processes this data can still introduce bias.

3. **Societal Bias**: This type of bias reflects broader societal prejudices and inequalities. AI systems, being products of human society, can inherit and perpetuate these biases if not carefully designed and monitored.

**Impact of Bias**
The impact of biased AI systems can be profound and far-reaching. Biased models can lead to discriminatory practices in areas such as hiring, lending, law enforcement, and healthcare. For instance, a biased hiring algorithm might unfairly favor candidates from certain demographic groups, leading to unequal employment opportunities.

Real-world examples of biased AI systems highlight the urgency of addressing this issue. For instance, a study found that a widely used healthcare algorithm disproportionately favored white patients over black patients when determining eligibility for certain health programs, leading to disparities in care.



### Introduction to AI Fairness 360

AI Fairness 360 (AIF360) is an open-source toolkit developed by IBM to address the challenge of bias in machine learning models. The toolkit provides a comprehensive suite of metrics to test for biases and algorithms to mitigate them.

**Toolkit Overview**
AIF360 includes:

- **Datasets**: A variety of datasets commonly used in fairness research, such as the Adult Income dataset and the COMPAS dataset.
- **Metrics**: A collection of fairness metrics to evaluate bias in datasets and models, including disparate impact, statistical parity, and equal opportunity difference.
- **Algorithms**: Several bias mitigation algorithms, such as reweighing, prejudice remover, and adversarial debiasing.

**Installation**
To get started with AI Fairness 360, you first need to install the toolkit. You can install it using pip:

```python
!pip install aif360
```

In the following sections, we will walk through a practical example using AIF360 to detect and mitigate bias in a machine learning model.



---

### Practical Example: Detecting and Mitigating Bias




In this example, we will use the Recidivism for Offenders Released from Prison dataset [directly from the IOWA Open Data Portal](https://data.iowa.gov/Correctional-System/Iowa-Prison-Recidivism-Status/akzb-ddk8/about_data), which provides information on individuals released from prison and their likelihood of reoffending. This dataset is suitable for studying fairness as it contains various demographic features that can be analyzed for bias.


**Dataset Construction**
To use AIFairness 360, the data must be converted into a proprietary AIF360 dataset object. The requirements for the Dataset objects are as follows:
1. Categorical Features encoded numerically (Not One hot encoded).
2. No null values.
3. Defined **Protected attributes** (e.g. Race, Sex).
    - **Definition**: Attributes that refer to characteristics of individuals that are legally protected against discrimination. These can include race, gender, age, religion, disability status, and other similar attributes.
4. For each protected attribute, defined Privileged groups (e.g. Male or White) and Unpriviledged Groups (e.g. Female, Non-white). 
    - **Privileged Groups**: Subsets of individuals within a dataset that have historically been favored or have had advantages in societal contexts based on certain protected attributes.
    - **Unprivileged Groups**: Subsets of individuals within a dataset that have historically been disadvantaged or have faced discrimination based on certain protected attributes.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", 100)
df = pd.read_csv("blog_post/data/Iowa_Prison_Recidivism_Status_20240724.csv", 
                 index_col=0, usecols=range(0, 23-7))   

## Quick Conversion of Dtypes for Clean Data
df = df.convert_dtypes(convert_string=False)
df.info()
df.head()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn import set_config

# Set sklearn to return a dataframe
set_config(transform_output='pandas')

# Use SimpleImputer to fill missing values with the most frequent value
imp = SimpleImputer(strategy="most_frequent")
df = imp.fit_transform(df)
df.isna().sum()


#### Numerically Encode Race

In [ ]:
# Preview sensitive columns
df['Race'].value_counts()

In [ ]:
# Numerically encoding the sensitive column using rank ordering
race_map = dict(zip(df['Race'].unique(), range(len(df['Race'].unique()))))
race_map

In [ ]:
# Apply the mapping to encode the sensitive column
df['Race'] = df['Race'].map(race_map)
df['Race'].value_counts()

In [ ]:
# # Convert the DataFrame to an AIF360-compatible dataset
# privileged_groups = [{'Race': 0}]
# unprivileged_groups = [{"Race":i} for i in range(1,len(race_map))]
# unprivileged_groups

### Numerically Encode Sex

In [ ]:
df['Sex'].value_counts()

In [ ]:
sex_map = {"Male":0,"Female":1}
df["Sex"] = df['Sex'].map(sex_map)
df['Sex'].value_counts()

### 👉Defining the Protected Attributes and Priveledged/Unpriveldged Groups

In [ ]:
# Convert the DataFrame to an AIF360-compatible dataset
protected_attributes= [ 'Race','Sex']
privileged_groups = [{'Race': 0}, {"Sex":0}]
unprivileged_groups = [{"Race":i} for i in range(1,len(race_map))] + [{"Sex":1}]
unprivileged_groups

# # Convert the DataFrame to an AIF360-compatible dataset
# protected_attributes= [ 'Sex']
# privileged_groups = [{"Sex":0}]
# unprivileged_groups = [{"Sex":1}]
# unprivileged_groups

#### Construct the AIF360 Dataset

In [ ]:
df.isna().sum()

In [ ]:
df

In [ ]:
df.columns

In [ ]:
# Remaining categorical features must be encoded
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(drop=None, sparse_output=False)

# Drop unnecessary columns 
drop_cols = ['Supervising Unit','Supervision Start Date','Supervision End Date']

# Categorical columns to be encoded
cat_cols=  ['Supervision Type','Supervision End Reason',"Supervision End Reason",'Supervision Offense Class','Supervision Offense Type',
       'Supervision Offense Subtype']


ohe_df = encoder.fit_transform(df[cat_cols])

# Concatenate one-hot encoded columns with the original DataFrame
# final_df = pd.concat([df.drop(columns=[*cat_cols, *drop_cols]), ohe_df],axis=1)
final_df = pd.get_dummies(df.drop(columns=drop_cols), columns=cat_cols)
final_df

In [ ]:
final_df.info()

In [ ]:
obj_cols = final_df.select_dtypes('object').columns
obj_cols

In [ ]:
final_df[obj_cols] = final_df[obj_cols].astype(float)

# BOOKMARK: DENTIST
### Comparison of `BinaryLabelDataset` and `StandardDataset` Classes in AI Fairness 360

The AI Fairness 360 (AIF360) toolkit provides different dataset classes to handle various types of data structures and use cases. Two commonly used classes are `BinaryLabelDataset` and `StandardDataset`. Below is a detailed comparison and contrast of these two classes.

#### BinaryLabelDataset

**Purpose**: Designed specifically for binary classification tasks where the target variable (label) has only two possible outcomes.

**Key Features**:
- **Label Handling**: Assumes that the label (target variable) has binary values (e.g., 0 and 1, "yes" and "no").
- **Protected Attributes**: Handles protected attributes for fairness analysis.
- **Metrics Compatibility**: Provides compatibility with fairness metrics that are specific to binary classification tasks.
- **Examples**: Suitable for datasets like the Adult Income dataset, where the task is to predict if income is above or below $50K.

**Example Usage**:
```python
from aif360.datasets import BinaryLabelDataset

dataset = BinaryLabelDataset(df, label_name='income', favorable_classes=[1], protected_attribute_names=['race'])
```

**Advantages**:
- Simplified handling of binary labels.
- Directly compatible with binary classification metrics in AIF360.

**Limitations**:
- Limited to binary classification tasks.
- Requires preprocessing if the dataset has multi-class labels or other complexities.

#### StandardDataset

**Purpose**: A more general-purpose dataset class that can handle a wide variety of data structures, including both binary and multi-class classification tasks.

**Key Features**:
- **Label Handling**: Can handle binary, multi-class, and continuous labels.
- **Flexibility**: More flexible in terms of the types of data it can represent.
- **Protected Attributes**: Handles protected attributes for fairness analysis.
- **Customizability**: Provides more options for customization, including handling multiple protected attributes and various types of labels.
- **Examples**: Suitable for datasets like the COMPAS dataset (recidivism prediction), where the task could involve multiple classes or continuous labels.

**Example Usage**:
```python
from aif360.datasets import StandardDataset

dataset = StandardDataset(df, label_name='recidivism', favorable_classes=[0], protected_attribute_names=['race'], privileged_classes=[[1]])
```

**Advantages**:
- Greater flexibility and customizability.
- Suitable for a broader range of tasks beyond binary classification, including multi-class and regression tasks.
- Allows for more complex preprocessing and handling of different types of labels.

**Limitations**:
- More complex to set up and use compared to `BinaryLabelDataset`.
- May require additional customization and configuration for specific use cases.

### Key Differences

1. **Target Use Case**:
   - `BinaryLabelDataset`: Specifically designed for binary classification tasks.
   - `StandardDataset`: General-purpose, suitable for binary, multi-class, and continuous label tasks.

2. **Label Handling**:
   - `BinaryLabelDataset`: Assumes binary labels.
   - `StandardDataset`: Can handle binary, multi-class, and continuous labels.

3. **Flexibility**:
   - `BinaryLabelDataset`: Less flexible, optimized for simplicity in binary classification.
   - `StandardDataset`: More flexible, can handle complex datasets with various types of labels and multiple protected attributes.

4. **Setup and Configuration**:
   - `BinaryLabelDataset`: Easier and quicker to set up for binary classification tasks.
   - `StandardDataset`: Requires more setup and customization but offers greater versatility.

### Conclusion

The choice between `BinaryLabelDataset` and `StandardDataset` depends on the specific requirements of your project. If you are working on a straightforward binary classification task, `BinaryLabelDataset` provides a simpler and more streamlined option. However, if your project involves more complex data structures, such as multi-class labels or multiple protected attributes, `StandardDataset` offers the flexibility and customizability needed to handle these complexities effectively.

In [ ]:
from aif360.datasets import StandardDataset, BinaryLabelDataset

# Assuming 'label' is your target column and 'gender' is your protected attribute

binary_dataset = BinaryLabelDataset(
        df=final_df,
        label_names=["Reincarcerated"],
        protected_attribute_names = protected_attributes,
        # privileged_groups = privileged_groups,
        # favorable_label=[{"Race":0],
        # unfavorable_label=1
    )
binary_dataset
from aif360.datasets import BinaryLabelDataset

# Create a BinaryLabelDataset with multiple protected attributes
binary_dataset = BinaryLabelDataset(
    df,
    label_name='recidivism',
    favorable_classes=[0],  # Non-recidivism is favorable
    protected_attribute_names=['race', 'gender'],
    privileged_classes=[[1], [1]]  # Privileged groups: White and Male
)

# Print dataset details
print(binary_dataset.feature_names)
print(binary_dataset.label_names)
print(binary_dataset.protected_attribute_names)

# target_col = "Reincarcerated"
# # Creating a data dictionary like the one used in AIF360's medical expenditure example
# data_dict={'feature_names':df.drop(coluns=target_col).columns,  # Question: Use transofrmed or original??
#            'label_names':[target_col], 
#            'protected_attribute_names': ["Sex"],
#            'privileged_protected_attributes': [np.array([1])], 
#            'unprivileged_protected_attributes': [np.array([0])],
#            }


The `BinaryLabelDataset` class in AI Fairness 360 is a specialized subclass of `StandardDataset` designed specifically for binary classification tasks. Because `BinaryLabelDataset` is built on top of `StandardDataset`, it inherits all its functionalities and adds some specific conveniences for handling binary labels. Therefore, there is nothing inherently unique that `BinaryLabelDataset` can do that `StandardDataset` cannot. However, `BinaryLabelDataset` provides some convenience features and simplified handling specifically for binary classification tasks. Here’s a detailed look:

### Convenience Features of `BinaryLabelDataset`

1. **Simplified Initialization**:
   - **Default Configurations**: `BinaryLabelDataset` comes with default configurations optimized for binary classification, making it easier to set up for such tasks without needing extensive customization.
   - **Example**:
     ```python
     from aif360.datasets import BinaryLabelDataset

     dataset = BinaryLabelDataset(df, label_name='income', favorable_classes=[1], protected_attribute_names=['race'])
     ```

2. **Preconfigured Metrics Compatibility**:
   - **Metric Calculation**: When using `BinaryLabelDataset`, it’s straightforward to calculate fairness metrics specific to binary classification. These metrics, such as disparate impact or statistical parity difference, are preconfigured for binary outcomes.
   - **Example**:
     ```python
     from aif360.metrics import BinaryLabelDatasetMetric

     metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{'race': 1}], unprivileged_groups=[{'race': 0}])
     print(f"Disparate Impact: {metric.disparate_impact()}")
     ```

3. **Assumptions for Binary Labels**:
   - **Favorable and Unfavorable Classes**: `BinaryLabelDataset` explicitly handles the designation of favorable and unfavorable classes, streamlining tasks where such distinctions are important.
   - **Example**:
     ```python
     dataset = BinaryLabelDataset(df, label_name='income', favorable_classes=[1], protected_attribute_names=['race'])
     ```

### Capabilities of `StandardDataset`

`StandardDataset` is more general and can handle a variety of data structures, including:

1. **Multi-Class Classification**:
   - **Multiple Labels**: `StandardDataset` can be used for tasks involving more than two classes, offering greater flexibility for datasets with multi-class labels.
   - **Example**:
     ```python
     from aif360.datasets import StandardDataset

     dataset = StandardDataset(df, label_name='health_status', favorable_classes=['healthy'], protected_attribute_names=['race'])
     ```

2. **Continuous Labels**:
   - **Regression Tasks**: `StandardDataset` can handle datasets where the target variable is continuous, making it suitable for regression tasks.
   - **Example**:
     ```python
     dataset = StandardDataset(df, label_name='income', favorable_classes=[lambda x: x > 50000], protected_attribute_names=['race'])
     ```

3. **Custom Preprocessing**:
   - **Flexible Configuration**: `StandardDataset` allows for more complex preprocessing and configuration, accommodating a wide range of use cases beyond binary classification.

### Conclusion

While `BinaryLabelDataset` offers a streamlined, convenient interface specifically for binary classification tasks, anything it can do can also be achieved with `StandardDataset` through additional customization. The main advantage of using `BinaryLabelDataset` lies in its ease of use and the fact that it simplifies certain aspects of handling binary labels, such as setting up fairness metrics and defining favorable/unfavorable classes. However, for more complex tasks or different types of labels, `StandardDataset` is the more versatile and flexible option.

In [ ]:
df.head()
df.info()

In [ ]:
# # Define the unprivileged and privileged groups
# privileged_groups =[ ]
# unprivileged_groups = [ ]

# for sens_attr in data_dict['protected_attribute_names']:
    
#     temp = data_dict['privileged_protected_attributes']
#     privileged_groups.append({sens_attr: temp[0][0]})
    
#     temp = data_dict['unprivileged_protected_attributes']
#     unprivileged_groups.append({sens_attr: temp[0][0]})
# privileged_groups, unprivileged_groups


**Exploring the Dataset**
Before diving into bias detection, it's important to understand the dataset. The Adult Income dataset includes features such as age, education, occupation, race, and gender.



```python
print(dataset.features)
print(dataset.labels)
```



**Detecting Bias**
Next, we will use AI Fairness 360 to detect bias in the dataset. We will start by calculating the disparate impact, a commonly used fairness metric.



```python
from aif360.metrics import BinaryLabelDatasetMetric

metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{'race': 1}], unprivileged_groups=[{'race': 0}])
print(f"Disparate Impact: {metric.disparate_impact()}")
```



**Mitigating Bias**
After detecting bias, the next step is to mitigate it. We will use the reweighing algorithm to adjust the weights of different groups in the dataset to ensure fairer outcomes.



```python
from aif360.algorithms.preprocessing import Reweighing

rw = Reweighing(unprivileged_groups=[{'race': 0}], privileged_groups=[{'race': 1}])
dataset_transf = rw.fit_transform(dataset)
```



**Evaluating Results**
Finally, we will evaluate the effectiveness of our bias mitigation efforts by comparing the disparate impact metric before and after applying the reweighing algorithm.

```python
metric_transf = BinaryLabelDatasetMetric(dataset_transf, privileged_groups=[{'race': 1}], unprivileged_groups=[{'race': 0}])
print(f"Disparate Impact after mitigation: {metric_transf.disparate_impact()}")
```



### Conclusion

In this lesson, we explored the importance of fairness in AI and how the AI Fairness 360 toolkit can help detect and mitigate bias in machine learning models. By ensuring our AI systems operate fairly, we can create more equitable and just outcomes for all.



### Further Reading
- IBM AI Fairness 360: [AIF360 Documentation](https://aif360.mybluemix.net/)
- Fairness and Machine Learning by Solon Barocas, Moritz Hardt, and Arvind Narayanan
- Algorithmic Bias Detection and Mitigation: Best Practices and Policies to Reduce Consumer Harms by the Federal Trade Commission

### Call to Action
I encourage you to experiment with AI Fairness 360 in your projects and contribute to creating fair AI systems. Together, we can make AI work for everyone.

### References
- [Disparate Impact](https://en.wikipedia.org/wiki/Disparate_impact)
- [AI Fairness 360 GitHub Repository](https://github.com/IBM/AIF360)
- [Bias in AI: A Review of Literature](https://arxiv.org/abs/1908.09635)
___

## Glossary for AI Fairness 360 Terms

#### Protected Attributes
**Definition**: Attributes that refer to characteristics of individuals that are legally protected against discrimination. These can include race, gender, age, religion, disability status, and other similar attributes.
**Example**: In a dataset about job applicants, 'race' and 'gender' could be considered protected attributes.

#### Privileged Groups
**Definition**: Subsets of individuals within a dataset that have historically been favored or have had advantages in societal contexts based on certain protected attributes.
**Example**: In the context of gender, males may be considered a privileged group in certain employment datasets.

#### Unprivileged Groups
**Definition**: Subsets of individuals within a dataset that have historically been disadvantaged or have faced discrimination based on certain protected attributes.
**Example**: In the context of race, individuals identifying as Black or Hispanic might be considered unprivileged groups in certain societal contexts.

#### Favorable Label
**Definition**: The outcome or class in a dataset that is considered positive or beneficial for the individual.
**Example**: In a dataset predicting loan approvals, a favorable label would be 'approved' (indicating that the loan application was successful).

#### Unfavorable Label
**Definition**: The outcome or class in a dataset that is considered negative or detrimental for the individual.
**Example**: In a dataset predicting recidivism, an unfavorable label would be 'reoffended' (indicating that the individual reoffended after being released from prison).

### Additional Similar Terms for AIF360

#### Bias Mitigation
**Definition**: The process of reducing or eliminating bias in machine learning models. This can involve techniques and algorithms that adjust the data or model to achieve fairer outcomes.
**Example**: Reweighing, which adjusts the weights of different groups in the training data, is a bias mitigation technique.

#### Disparate Impact
**Definition**: A measure used to determine if a decision-making process disproportionately affects a particular protected group. It is calculated as the ratio of the rate of a favorable outcome for the unprivileged group to the rate of a favorable outcome for the privileged group.
**Example**: If the hiring rate for women (unprivileged group) is 30% and for men (privileged group) is 60%, the disparate impact is 0.5 (30% / 60%).

#### Fairness Metrics
**Definition**: Quantitative measures used to assess the fairness of a machine learning model. These metrics evaluate how equally outcomes are distributed among different groups based on protected attributes.
**Example**: Statistical parity, equal opportunity difference, and disparate impact are examples of fairness metrics.

#### Bias Detection
**Definition**: The process of identifying bias in datasets or machine learning models. This involves analyzing the data and model predictions to uncover patterns of unfair treatment based on protected attributes.
**Example**: Calculating the disparate impact ratio to check if a model's predictions are biased against a particular racial group.

#### Adversarial Debiasing
**Definition**: A bias mitigation technique where an adversarial model is trained to predict the protected attribute from the original model’s predictions. The goal is to adjust the original model to make it difficult for the adversarial model to correctly predict the protected attribute, thus reducing bias.
**Example**: Training an adversarial network alongside the main model to ensure that the predictions are not correlated with the protected attributes.

---

This glossary should help your readers understand the key concepts and terminology associated with AI Fairness 360. If you have any more terms to include or need further details, let me know!